In [1]:
import pandas as pd 
import numpy as np
import os 
import glob

# previous work

In [ ]:
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
aseg_path = os.path.join(subj_dir, "stats/aseg.stats")
metrics = ["NVoxels", "Volume_mm3", "normMean", "normStdDev", "normMin", "normMax", "normRange"]

#read in data
aseg_data = pd.read_csv(
    aseg_path,
    sep='\s+',
    comment="#"
)

#drop the first 3 columns (as these contain no relevant info)
aseg_data = aseg_data.drop(columns = aseg_data.iloc[:, range(2)])
aseg_data.columns = [
    "NVoxels", "Volume_mm3", "StructName",
    "normMean", "normStdDev", "normMin", "normMax", "normRange"
]

#the columns are now:
#NVoxels Volume_mm3 StructName normMean normStdDev normMin normMax normRange 

#pivot data with the structname
pivoted_data = aseg_data.pivot_table(
    index= "StructName",
    values= metrics
)

#transpose data
pivoted_data = pivoted_data.T

#add subject id 
subject_id = os.path.basename(os.path.normpath(subj_dir))
pivoted_data.columns = pd.MultiIndex.from_product([[subject_id], pivoted_data.columns])

print(pivoted_data)
print(pivoted_data.shape)

In [8]:
#code to extract the #measure metrics for one participant
#should probably include this in the rh_ code but can also extract is separately and then merge the panda's dataframes
#now works but need to loop over all subjects for it to actually work
import pandas as pd
import re

root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
aseg_path = os.path.join(subj_dir, "stats/aseg.stats")


target_metrics = [
    "BrainSegVol",
    "BrainSegVolNotVent",
    "BrainSegVolNotVentSurf",
    "VentricleChoroidVol",
    "lhCortexVol", 
    "rhCortexVol",
    "CortexVol",
    "lhCerebralWhiteMatterVol",
    "rhCerebralWhiteMatterVol",
    "CerebralWhiteMatterVol", 
    "SubCortGrayVol", 
    "TotalGrayVol",
    "SupraTentorialVol",
    "SupraTentorialVolNotVent",
    "SupraTentorialVolNotVentVox",
    "MaskVol", 
    "BrainSegVol-to-eTIV", 
    "MaskVol-to-eTIV",
    "lhSurfaceHoles",
    "rhSurfaceHoles", 
    "SurfaceHoles", 
    "eTIV"#, 
    #"avg_thickness"
]

metrics_dict = {}
with open(aseg_path, "r") as file:
    for line in file:
        if line.startswith("# Measure"):
            match = re.match(r"# Measure (\w+), ([\w\-\.]+), .*?, (\d+\.\d+|\d+),", line)
            #match = re.match(r"# Measure (\w+), ([\w\-\.]+), .*?, (\d+\.\d+|\d+|\.d+),", line)
           
            if match:
                metric_name = match.group(2)
                metric_value = match.group(3)
                if metric_name in target_metrics:
                    metrics_dict[metric_name] = metric_value

metrics_df = pd.DataFrame([metrics_dict])

subject_id = subj_dir.split("/")[-1]  
metrics_df["SubjectID"] = subject_id

metrics_df = metrics_df[["SubjectID"] + [m for m in target_metrics if m in metrics_dict]]
metrics_df = metrics_df.set_index("SubjectID")
print(metrics_df)
print(metrics_df.shape)


              BrainSegVol BrainSegVolNotVent BrainSegVolNotVentSurf  \
SubjectID                                                             
1533138    1109928.000000     1091828.000000         1091560.264705   

          VentricleChoroidVol    lhCortexVol    rhCortexVol      CortexVol  \
SubjectID                                                                    
1533138          14644.000000  239995.398460  230734.026454  470729.424915   

          lhCerebralWhiteMatterVol rhCerebralWhiteMatterVol  \
SubjectID                                                     
1533138              216804.832342            214283.007449   

          CerebralWhiteMatterVol SubCortGrayVol   TotalGrayVol  \
SubjectID                                                        
1533138            431087.839791   51003.000000  632310.424915   

          SupraTentorialVol SupraTentorialVolNotVent  \
SubjectID                                              
1533138       972366.264705            957722.2647

In [10]:
#code to extract the # Measure from aseg.stats and put into a dataframe
import os
import re
import pandas as pd

root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"
all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_metrics = pd.DataFrame()

target_metrics = [
        "BrainSegVol",
        "BrainSegVolNotVent",
        "BrainSegVolNotVentSurf",
        "VentricleChoroidVol",
        "lhCortexVol", 
        "rhCortexVol",
        "CortexVol",
        "lhCerebralWhiteMatterVol",
        "rhCerebralWhiteMatterVol",
        "CerebralWhiteMatterVol", 
        "SubCortGrayVol", 
        "TotalGrayVol",
        "SupraTentorialVol",
        "SupraTentorialVolNotVent",
        "SupraTentorialVolNotVentVox",
        "MaskVol", 
        "BrainSegVol-to-eTIV", 
        "MaskVol-to-eTIV",
        "lhSurfaceHoles",
        "rhSurfaceHoles", 
        "SurfaceHoles", 
        "eTIV"#, 
        #"avg_thickness"
        ]

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    aseg_path = os.path.join(subj_dir, "stats", "aseg.stats")

    if os.path.exists(aseg_path):
        metrics_dict = {}
        with open(aseg_path, "r") as file:
            for line in file:
                if line.startswith("# Measure"):
                    match = re.match(r"# Measure (\w+), ([\w\-\.]+), .*?, (\d+\.\d+|\d+),", line)
        
                    if match:
                        metric_name = match.group(2)
                        metric_value = match.group(3)
                        if metric_name in target_metrics:
                            metrics_dict[metric_name] = metric_value
        
        metrics_df = pd.DataFrame([metrics_dict])
        
        subject_id = subj_dir.split("/")[-1]  
        metrics_df["SubjectID"] = subject_id
        
        metrics_df = metrics_df[["SubjectID"] + [m for m in target_metrics if m in metrics_dict]]
        all_metrics = pd.concat([all_metrics, metrics_df], ignore_index=True)

final_df = all_metrics.set_index("SubjectID")

print("\nFinal combined DataFrame:")
print(final_df.tail())
#final_df.to_csv("measures_aseg_stats.csv")


Final combined DataFrame:
              BrainSegVol BrainSegVolNotVent BrainSegVolNotVentSurf  \
SubjectID                                                             
3463731    1087763.000000     1074155.000000         1074044.590841   
3814047    1381218.000000     1361802.000000         1361118.069372   
4851516    1171168.000000     1126991.000000         1126172.653085   
3626816    1312016.000000     1296802.000000         1296479.157079   
2420905    1212468.000000     1131306.000000         1129281.415336   

          VentricleChoroidVol    lhCortexVol    rhCortexVol      CortexVol  \
SubjectID                                                                    
3463731          10821.000000  223084.305442  223427.153869  446511.459311   
3814047          15852.000000  267234.026324  268869.745404  536103.771728   
4851516          39066.000000  234958.938990  243924.327341  478883.266331   
3626816          13016.000000  284966.233468  281963.022961  566929.256429   
2420905

In [14]:
print("\nFinal combined DataFrame:")
print(final_df.head())
print(final_df.shape)


Final combined DataFrame:
              BrainSegVol BrainSegVolNotVent BrainSegVolNotVentSurf  \
SubjectID                                                             
1533138    1109928.000000     1091828.000000         1091560.264705   
1875529    1127087.000000     1079149.000000         1078238.559246   
4758824    1140990.000000     1109558.000000         1109367.068707   
5304207    1119328.000000     1102891.000000         1102308.359285   
2747983    1177154.000000     1157188.000000         1157129.506938   

          VentricleChoroidVol    lhCortexVol    rhCortexVol      CortexVol  \
SubjectID                                                                    
1533138          14644.000000  239995.398460  230734.026454  470729.424915   
1875529          41059.000000  238784.863444  214459.362318  453244.225762   
4758824          27856.000000  229865.245490  244370.530076  474235.775566   
5304207          13353.000000  222587.937796  228103.351494  450691.289290   
2747983

In [4]:
#aseg code to extract the columns of dataframe [already used and extracted]
import os
import re
import pandas as pd

root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"
all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = []

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    aseg_path = os.path.join(subj_dir, "stats", "aseg.stats")

    if os.path.exists(aseg_path):        
        aseg_data = pd.read_csv(aseg_path, sep=r"\s+", comment="#", header=None)
        aseg_data = aseg_data.drop(aseg_data.columns[:2], axis=1)

        aseg_data.columns = [
            "NVoxels", "Volume_mm3", "StructName",
            "normMean", "normStdDev", "normMin", "normMax", "normRange"
        ]

        pivoted_data = aseg_data.pivot_table(
            index=None,
            columns="StructName",
            values= ["normMean", "normStdDev"] #, "normMin", "normMax", "normRange"] 
        )

        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        all_data.append(pivoted_data)

final_df = pd.concat(all_data, axis=0)

print("\nFinal combined DataFrame:")
print(final_df.head())

#final_df.to_csv("combined_aseg_stats.csv")



Final combined DataFrame:
StructName  3rd-Ventricle  4th-Ventricle  5th-Ventricle  Brain-Stem  \
SubjectID                                                             
1533138           36.8674        31.3818            0.0     81.5837   
1533138           11.0437         9.4151            0.0      9.5555   
1875529           31.3638        30.1527            0.0     83.1511   
1875529            9.4712        10.0800            0.0      9.6401   
4758824           35.0732        36.2819            0.0     84.1654   

StructName  CC_Anterior  CC_Central  CC_Mid_Anterior  CC_Mid_Posterior  \
SubjectID                                                                
1533138        100.2311     94.2112          93.9213           89.7624   
1533138         17.1138     17.2545          18.2093           18.9696   
1875529         95.3329     89.9717          90.3416           85.7553   
1875529         16.4144     17.5802          17.7160           19.0117   
4758824         99.6201     95.

In [6]:
print(final_df.shape)

(75900, 47)


final_df.to_csv("combined_aseg_stats.csv")

# Now let's create the dataset for the right and left hand side:


/project_cephfs/3022017.06/UKB/freesurfer/1533138/stats/lh.aparc.a2009s.stats
/project_cephfs/3022017.06/UKB/freesurfer/1533138/stats/rh.aparc.a2009s.stats



ColHeaders StructName NumVert SurfArea GrayVol ThickAvg ThickStd MeanCurv GausCurv FoldInd CurvInd
G&S_frontomargin                          964    697   2432  2.884 0.737     0.165     0.052       16     2.5


In [25]:
#LH_ file for 1 subject
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
lh_path = os.path.join(subj_dir, "stats","lh.aparc.a2009s.stats")
metrics = ["ColHeaders", "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

#read in data
lh_data = pd.read_csv(
    lh_path,
    sep='\s+',
    comment="#"
)

lh_data.columns = [
    "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"
]

#pivot data with the structname
pivoted_data = lh_data.pivot_table(
    index= "StructName",
    values= ["ThickAvg"] #,"ThickStd"]
)

#transpose data
pivoted_data = pivoted_data.T
pivoted_data = pivoted_data.add_prefix("lh_")
pivoted_data = pivoted_data.add_suffix("_thickness")

#add subject id 
subject_id = os.path.basename(os.path.normpath(subj_dir))
pivoted_data.columns = pd.MultiIndex.from_product([[subject_id], pivoted_data.columns])
pivoted_data["SubjectID"] = subject_id

pivoted_data = pivoted_data.set_index("SubjectID")
print(pivoted_data)

                               1533138                                  \
StructName lh_G&S_cingul-Ant_thickness lh_G&S_cingul-Mid-Ant_thickness   
SubjectID                                                                
1533138                          3.217                           3.067   

                                                                            \
StructName lh_G&S_cingul-Mid-Post_thickness lh_G&S_occipital_inf_thickness   
SubjectID                                                                    
1533138                               2.777                           2.89   

                                                                     \
StructName lh_G&S_paracentral_thickness lh_G&S_subcentral_thickness   
SubjectID                                                             
1533138                            2.41                       2.989   

                                                                               \
StructName lh_G&S_transv

<>:12: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\s'
/scratch/quirom/slurm_job_49464438/ipykernel_3333150/868605765.py:12: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',


In [ ]:
#for rh for 1 subj
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
rh_path = os.path.join(subj_dir, "stats","rh.aparc.a2009s.stats")
metrics = ["ColHeaders", "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

#read in data
rh_data = pd.read_csv(
    rh_path,
    sep='\s+',
    comment="#"
)

rh_data.columns = [
    "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"
]

#pivot data with the structname
pivoted_data = rh_data.pivot_table(
    index= "StructName",
    values= ["ThickAvg","ThickStd"]
)

#transpose data
pivoted_data = pivoted_data.T
pivoted_data = pivoted_data.add_prefix("rh_")
pivoted_data = pivoted_data.add_suffix("_thickness")


#add subject id 
subject_id = os.path.basename(os.path.normpath(subj_dir))
pivoted_data.columns = pd.MultiIndex.from_product([[subject_id], pivoted_data.columns])

print(pivoted_data)
print(pivoted_data.shape)

In [26]:
#loop over all LH files and create CSV file
import os
import re
import pandas as pd
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = pd.DataFrame()

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    lh_path = os.path.join(subj_dir, "stats", "lh.aparc.a2009s.stats")

    if os.path.exists(lh_path):        
        lh_data = pd.read_csv(lh_path, sep=r"\s+", comment="#", header=None)
        lh_data.columns = ["StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

        pivoted_data = lh_data.pivot_table(
            index="StructName",
            columns=None,
            values= ["ThickAvg"] #,"ThickStd"]
        )

        #transpose data
        pivoted_data = pivoted_data.T
        pivoted_data = pivoted_data.add_prefix("lh_")
        pivoted_data = pivoted_data.add_suffix("_thickness")

        #set subj as index
        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        #add to dataframes
        all_data = pd.concat([all_data, pivoted_data])


print("\nFinal combined DataFrame:")
print(all_data.head())
print(all_data.shape)

#all_data.to_csv("lh_stats.csv")



Final combined DataFrame:
StructName  lh_G&S_cingul-Ant_thickness  lh_G&S_cingul-Mid-Ant_thickness  \
SubjectID                                                                  
1533138                           3.217                            3.067   
1875529                           3.205                            3.207   
4758824                           2.734                            1.964   
5304207                           3.070                            2.655   
2747983                           3.072                            3.018   

StructName  lh_G&S_cingul-Mid-Post_thickness  lh_G&S_frontomargin_thickness  \
SubjectID                                                                     
1533138                                2.777                          2.884   
1875529                                2.756                          2.896   
4758824                                2.683                          2.311   
5304207                                2.868 

In [27]:
all_data.to_csv("lh_stats.csv")

In [28]:
#loop over RH files and create csv file
import os
import re
import pandas as pd
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = pd.DataFrame()

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    rh_path = os.path.join(subj_dir, "stats", "rh.aparc.a2009s.stats")

    if os.path.exists(rh_path):        
        rh_data = pd.read_csv(rh_path, sep=r"\s+", comment="#", header=None)
        rh_data.columns = ["StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

        pivoted_data = rh_data.pivot_table(
            index="StructName",
            columns=None,
            values= ["ThickAvg"] #,"ThickStd"]
        )

        #transpose data
        pivoted_data = pivoted_data.T
        pivoted_data = pivoted_data.add_prefix("rh_")
        pivoted_data = pivoted_data.add_suffix("_thickness")

        #set subj as index
        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        #add to dataframes
        all_data = pd.concat([all_data, pivoted_data])


print("\nFinal combined DataFrame:")
print(all_data.head())
print(all_data.shape)

#all_data.to_csv("rh_stats.csv")



Final combined DataFrame:
StructName  rh_G&S_cingul-Ant_thickness  rh_G&S_cingul-Mid-Ant_thickness  \
SubjectID                                                                  
1533138                           2.888                            2.826   
1875529                           2.835                            2.424   
4758824                           2.991                            3.015   
5304207                           2.932                            2.715   
2747983                           3.033                            3.070   

StructName  rh_G&S_cingul-Mid-Post_thickness  rh_G&S_frontomargin_thickness  \
SubjectID                                                                     
1533138                                2.626                          2.797   
1875529                                2.782                          2.287   
4758824                                2.864                          2.668   
5304207                                2.700 

all_data.to_csv("rh_stats.csv")

# Now we have the CSV files for LH, RH, Aseg.stats and Aseg measures

What i now need to do is:
1. drop the columns that do not contain any numbers (drop nan or something)
2. merge the dataframes with the "SubjectID" as a key --> google how to do this
4. merge this large datatable with the basic_demographics.csv
5. find site location in the data and add this to the large dataframe (with the SubjectID as a key)
6. compute euler (do more research and study the code given to you)
7. concat everything and you have the datatable

In [52]:
#54 is assessment centre
#53 is date of attending
# df_misc = pd.read_csv("/project_cephfs/3022017.05/phenotypes/current/99_miscellaneous.csv", nrows = 10000)
# cols = list(df_misc.columns)
# df_misc = df_misc.drop(columns = df_misc.iloc[:, range(13, len(cols))])
# df_misc = df_misc.drop(columns = df_misc.iloc[:, range(1,6)])
# print(df_misc)

filepath = "/project_cephfs/3022017.05/phenotypes/current/99_miscellaneous.csv"
chunksize = 10000
df_first = pd.read_csv(filepath, nrows=chunksize)

#strip irrelevant cols
cols = list(df_first.columns)
df_first = df_first.drop(columns=df_first.iloc[:, range(13, len(cols))])
df_first = df_first.drop(columns=df_first.iloc[:, range(1, 6)])
keep_cols = df_first.columns.tolist()

dfs = []
for chunk in pd.read_csv(filepath, chunksize=10000):
    df_clean = chunk[keep_cols]
    dfs.append(df_clean)
df_full = pd.concat(dfs, ignore_index=True)



In [53]:
df_full = df_full.rename(columns={"eid" : "SubjectID",
                                  "53-0.0" : "Year_scan_initial",
                                  "53-1.0" : "Year_scan_1_repeat",
                                  "53-2.0" : "Year_scan_imaging_visit",
                                  "54-0.0" : "Assessment_centre_initial",
                                  "54-2.0" : "Assessment_centre_1_repeat",
                                  "55-0.0" : "Month_scan_initial",
                                  "55-1.0" : "Month_scan_1_repeat",
                                 })

df_full = df_full.set_index("SubjectID")
#df_full.to_csv("Yr_mnth_site_scan.csv")

In [54]:
df_full.head()

,Year_scan_initial,Year_scan_1_repeat,Year_scan_imaging_visit,Assessment_centre_initial,Assessment_centre_1_repeat,Month_scan_initial,Month_scan_1_repeat
SubjectID,,,,,,,
1000012,2009.528767,NaN,NaN,11016.0,NaN,7.0,NaN
1000029,2009.857534,2013.032877,NaN,11014.0,NaN,11.0,1.0
1000031,2009.495890,NaN,NaN,11012.0,NaN,7.0,NaN
1000047,2008.229508,NaN,NaN,11005.0,NaN,3.0,NaN
1000050,2009.939726,NaN,2015.723288,11016.0,11025.0,12.0,NaN


In [58]:
#basic demographics 
#31 : sex (0: female, 1 : male)
#34 : birth year 

df_basic_demo = pd.read_csv("/project_cephfs/3022017.05/phenotypes/current/01_basic_demographics.csv")
df_basic_demo = df_basic_demo.drop(columns = df_basic_demo.iloc[:, range(3,4)])
df_basic_demo = df_basic_demo.rename(columns={"eid" : "SubjectID",
                                              "31-0.0" : "Sex",
                                              "34-0.0" : "Birthyear"
                                             })
df_basic_demo = df_basic_demo.set_index("SubjectID")
df_basic_demo
#df_basic_demo.to_csv("Sex_birthyear.csv")

,Sex,Birthyear
SubjectID,,
1000012,1.0,1946.0
1000029,0.0,1943.0
1000031,1.0,1958.0
1000047,1.0,1956.0
1000050,0.0,1944.0
...,...,...
6025263,1.0,1942.0
6025278,1.0,1955.0
6025280,1.0,1947.0


In [66]:
#check if specific participant is in the dataset
df_basic_demo.query('SubjectID == 1533138')

,Sex,Birthyear
SubjectID,,
1533138,0.0,1950.0


# Now we have the csv files that we need to calculate the age on the scan
What i now need to do is:
1. calculate the age of each participant during the scan
2. turn the year of scan into an integer
3. do year_scan - birthyear = age
How should i deal with the ages of the columns of the second scan? im not sure

In [1]:
import pandas as pd 
import os
import glob
import numpy as np
import re

In [5]:
#lets make the age column at first scan
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"
yr_mth_scan = os.path.join(csv_path, "Yr_mnth_site_scan.csv")
sex_birth = os.path.join(csv_path, "Sex_birthyear.csv")


df_yr_mnth = pd.read_csv(yr_mth_scan).set_index("SubjectID")
df_demo = pd.read_csv(sex_birth).set_index("SubjectID")


In [3]:
#turn to integer values
df_yr_mnth = df_yr_mnth.fillna(-1)
df_yr_mnth = df_yr_mnth.select_dtypes(include=['float']).astype(int)
df_yr_mnth

,Year_scan_initial,Year_scan_1_repeat,Year_scan_imaging_visit,Assessment_centre_initial,Assessment_centre_1_repeat,Month_scan_initial,Month_scan_1_repeat
SubjectID,,,,,,,
1000012,2009,-1,-1,11016,-1,7,-1
1000029,2009,2013,-1,11014,-1,11,1
1000031,2009,-1,-1,11012,-1,7,-1
1000047,2008,-1,-1,11005,-1,3,-1
1000050,2009,-1,2015,11016,11025,12,-1
...,...,...,...,...,...,...,...
6025263,2008,-1,-1,11010,-1,10,-1
6025278,2010,-1,-1,11014,-1,5,-1
6025280,2006,-1,2016,10003,-1,5,-1


In [4]:
#turn basic demo into ints
df_demo = df_demo.fillna(-1)
df_demo = df_demo.select_dtypes(include=['float']).astype(int)
df_demo

,Sex,Birthyear
SubjectID,,
1000012,1,1946
1000029,0,1943
1000031,1,1958
1000047,1,1956
1000050,0,1944
...,...,...
6025263,1,1942
6025278,1,1955
6025280,1,1947


In [20]:
df_merg = pd.merge(df_yr_mnth, df_demo, on="SubjectID")
df_merg["Age_during_scan"] = df_merg["Year_scan_initial"] - df_merg["Birthyear"]
df_merg = df_merg.fillna(-1)
df_merg["Age_during_2nd_scan"] = df_merg["Year_scan_imaging_visit"] - df_merg["Birthyear"]
df_merg




,Year_scan_initial,Year_scan_1_repeat,Year_scan_imaging_visit,Assessment_centre_initial,Assessment_centre_1_repeat,Month_scan_initial,Month_scan_1_repeat,Sex,Birthyear,Age_during_scan,Age_during_2nd_scan
SubjectID,,,,,,,,,,,
1000012,2009,-1,-1,11016,-1,7,-1,1,1946,63,-1947
1000029,2009,2013,-1,11014,-1,11,1,0,1943,66,-1944
1000031,2009,-1,-1,11012,-1,7,-1,1,1958,51,-1959
1000047,2008,-1,-1,11005,-1,3,-1,1,1956,52,-1957
1000050,2009,-1,2015,11016,11025,12,-1,0,1944,65,71
...,...,...,...,...,...,...,...,...,...,...,...
6025263,2008,-1,-1,11010,-1,10,-1,1,1942,66,-1943
6025278,2010,-1,-1,11014,-1,5,-1,1,1955,55,-1956
6025280,2006,-1,2016,10003,-1,5,-1,1,1947,59,69


In [24]:
df_merg.query('SubjectID == 2420905')

,Year_scan_initial,Year_scan_1_repeat,Year_scan_imaging_visit,Assessment_centre_initial,Assessment_centre_1_repeat,Month_scan_initial,Month_scan_1_repeat,Sex,Birthyear,Age_during_scan,Age_during_2nd_scan
SubjectID,,,,,,,,,,,
2420905,2009,-1,2016,11016,11025,7,-1,0,1949,60,67


In [56]:
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"
df = os.path.join(csv_path, "lh_stats.csv")
df = pd.read_csv(df)

mask = ~df["SubjectID"].astype(str).str.fullmatch(r"\d+")
non_mask = ~mask
amount = mask.sum() 
print(amount)

#mask = ~df["SubjectID"].astype(str).str.fullmatch(r"\d+")
#mask_other = df["SubjectID"].astype(str).str.contains(r'\d+_(?!long$|scan2$)')

#turns out there are 4008 scans that are either _long, _scan2
df.query("SubjectID == '4299969' or SubjectID == '4299969_long' or SubjectID == '4299969_scan2'")
df.dropna(axis=1)

4008


,SubjectID,lh_G_front_inf-Opercular_thickness,lh_G_front_inf-Triangul_thickness,lh_G_front_middle_thickness,lh_G_front_sup_thickness,lh_G_oc-temp_lat-fusifor_thickness,lh_G_oc-temp_med-Lingual_thickness,lh_G_oc-temp_med-Parahip_thickness,lh_G_occipital_middle_thickness,lh_G_orbital_thickness,...,lh_S_front_middle_thickness,lh_S_front_sup_thickness,lh_S_occipital_ant_thickness,lh_S_orbital-H_Shaped_thickness,lh_S_pericallosal_thickness,lh_S_postcentral_thickness,lh_S_precentral-inf-part_thickness,lh_S_precentral-sup-part_thickness,lh_S_temporal_inf_thickness,lh_S_temporal_sup_thickness
0,1533138,2.955,2.859,2.994,3.053,3.310,2.248,2.847,2.846,3.016,...,2.565,2.774,2.719,3.068,2.288,2.521,2.617,2.518,2.518,2.803
1,1875529,3.153,3.050,2.993,3.167,3.214,2.035,3.097,2.610,3.199,...,2.575,2.740,2.315,3.130,2.512,2.341,2.988,2.921,2.653,2.749
2,4758824,2.789,2.575,2.677,3.002,3.026,1.646,2.861,2.427,2.801,...,2.426,2.691,2.417,2.788,1.654,2.334,2.728,2.652,2.406,2.483
3,5304207,2.765,2.561,2.589,3.046,3.010,1.692,2.860,2.607,2.804,...,2.408,2.599,2.523,2.974,1.794,2.347,2.630,2.828,2.489,2.556
4,2747983,2.932,2.969,2.940,3.229,3.124,2.212,3.408,2.742,2.924,...,2.484,2.924,2.548,3.221,1.768,2.493,2.956,2.789,2.703,2.615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37951,3463731,2.984,2.743,2.868,3.188,2.943,2.110,3.555,2.797,3.222,...,2.507,2.780,2.715,3.190,1.574,2.510,3.044,2.823,2.658,2.616
37952,3814047,2.773,2.620,2.807,2.965,3.055,2.270,3.299,2.677,2.676,...,2.290,2.524,2.269,2.693,2.459,2.172,2.441,2.503,2.518,2.464
37953,4851516,2.861,2.838,2.639,2.796,2.808,2.121,3.124,2.659,2.616,...,2.309,2.516,2.511,2.831,1.827,2.149,2.524,2.486,2.118,2.644
37954,3626816,3.186,3.115,3.135,3.192,3.233,2.275,3.257,2.900,2.910,...,2.834,2.822,2.352,3.226,2.106,2.549,2.923,2.912,2.882,2.911


# any file that is _scan2 or _long uses the column "Year_scan_imaging_visit"
still need to drop those files in the original dataset

In [50]:
df_merg["Age_during_scan"] = df_merg["Year_scan_initial"] - df_merg["Birthyear"]
df_merg["Age_during_2nd_scan"] = df_merg["Year_scan_imaging_visit"] - df_merg["Birthyear"]
df_merg["Age"] = df_merg["Age_during_scan"]
df_merg.loc[df_merg["Age_during_2nd_scan"] > 0, "Age"] = df_merg["Age_during_2nd_scan"]
second_scans = df_merg[df_merg["Age_during_2nd_scan"] > 0].copy()
second_scans["ScanType"] = "SecondScan"
df_merg = df_merg.drop(second_scans.index)
df_merg["ScanType"] = "FirstScan"
combined_df = pd.concat([df_merg, second_scans], ignore_index=False)

combined_df


,Year_scan_initial,Year_scan_1_repeat,Year_scan_imaging_visit,Assessment_centre_initial,Assessment_centre_1_repeat,Month_scan_initial,Month_scan_1_repeat,Sex,Birthyear,Age_during_scan,Age_during_2nd_scan,Age,ScanType
SubjectID,,,,,,,,,,,,,
1000012,2009,-1,-1,11016,-1,7,-1,1,1946,63,-1947,63,FirstScan
1000029,2009,2013,-1,11014,-1,11,1,0,1943,66,-1944,66,FirstScan
1000031,2009,-1,-1,11012,-1,7,-1,1,1958,51,-1959,51,FirstScan
1000047,2008,-1,-1,11005,-1,3,-1,1,1956,52,-1957,52,FirstScan
1000068,2009,-1,-1,11016,-1,3,-1,0,1959,50,-1960,50,FirstScan
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6025242,2010,-1,-1,11021,-1,2,-1,0,1957,53,-1958,53,FirstScan
6025255,2009,-1,-1,11016,-1,6,-1,1,1950,59,-1951,59,FirstScan
6025263,2008,-1,-1,11010,-1,10,-1,1,1942,66,-1943,66,FirstScan


In [8]:
#let's create a large pandas dataframe with all the data we have up to this moment (we still need to figure out how to calculate the age, because something is going wrong
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"
data_1 = os.path.join(csv_path, "aseg_normmean_stats.csv")
data_2 = os.path.join(csv_path, "lh_stats.csv")
data_3 = os.path.join(csv_path, "rh_stats.csv")
data_4 = os.path.join(csv_path, "Yr_mnth_site_scan.csv")
data_5 = os.path.join(csv_path, "Sex_birthyear.csv")

df_1 = pd.read_csv(data_1)
df_2 = pd.read_csv(data_2)
df_3 = pd.read_csv(data_3)
df_4 = pd.read_csv(data_4)
df_5 = pd.read_csv(data_5)

df_1 = df_1.dropna(axis=1)
df_2 = df_2.dropna(axis=1)
df_3 = df_3.dropna(axis=1)
df_4 = df_4.dropna(axis=1)
df_5 = df_5.dropna(axis=1)

df_1 = df_1.reset_index()
df_2 = df_2.reset_index()
df_3 = df_3.reset_index()
df_4 = df_4.reset_index()
df_5 = df_5.reset_index()

#this now merges correctly!

combined_df = df_1.merge(df_2, on="SubjectID", how="left")
combined_df = combined_df.merge(df_3, on="SubjectID", how="left")

#still need to see how to combine the other dataframes --> maybe i need to ask where the correct files are if i cannot locate them myself
#also i should delete the _long and _scan2 files

#combined_df = combined_df.merge(df_4, on="SubjectID", how="left")
#combined_df = combined_df.merge(df_5, on="SubjectID", how="left")
#combined_df = combined_df.set_index("SubjectID")

drop_scans = combined_df[combined_df['SubjectID'].str.endswith('_scan2')].index
combined_df = combined_df.drop(drop_scans)

drop_scans = combined_df[combined_df['SubjectID'].str.endswith('_long')].index
combined_df = combined_df.drop(drop_scans)

combined_df.head(50)
# print(df_1.shape)
# print(df_2.shape)
# print(df_3.shape)


,index_x,SubjectID,3rd-Ventricle,4th-Ventricle,5th-Ventricle,Brain-Stem,CC_Anterior,CC_Central,CC_Mid_Anterior,CC_Mid_Posterior,...,rh_S_front_sup_thickness,rh_S_interm_prim-Jensen_thickness,rh_S_oc-temp_lat_thickness,rh_S_occipital_ant_thickness,rh_S_parieto_occipital_thickness,rh_S_pericallosal_thickness,rh_S_postcentral_thickness,rh_S_precentral-inf-part_thickness,rh_S_precentral-sup-part_thickness,rh_S_temporal_sup_thickness
0,0,1533138,36.8674,31.3818,0.0,81.5837,100.2311,94.2112,93.9213,89.7624,...,2.753,2.170,2.914,2.665,2.533,1.965,2.227,2.672,2.561,2.735
1,1,1875529,31.3638,30.1527,0.0,83.1511,95.3329,89.9717,90.3416,85.7553,...,2.481,2.309,2.913,2.403,2.287,2.407,2.016,2.794,2.706,2.676
2,2,4758824,35.0732,36.2819,0.0,84.1654,99.6201,95.3489,95.4416,93.4878,...,2.631,2.253,2.494,2.663,2.461,1.793,2.397,2.537,2.547,2.684
3,3,5304207,41.5897,33.3825,0.0,84.4920,104.1299,93.2735,95.2359,95.4301,...,2.644,2.580,2.840,2.850,2.390,1.782,2.377,2.501,2.696,2.746
4,4,2747983,42.9063,33.2600,0.0,83.8396,103.1391,92.3403,90.9298,94.4794,...,2.774,2.511,2.972,2.727,2.504,1.927,2.579,2.863,2.843,2.829
5,5,5279838,57.7897,34.9515,0.0,82.4166,104.4810,97.8531,95.3256,96.0647,...,2.692,2.350,2.537,2.767,2.407,1.578,2.278,2.646,2.671,2.743
6,6,4959500,34.7429,32.5413,0.0,83.6770,103.8699,94.1685,94.9715,93.6816,...,2.863,2.901,2.820,2.670,2.890,2.017,2.509,2.761,2.893,3.027
7,7,1206189,35.4566,34.7022,0.0,83.7808,98.1308,93.2479,93.6054,85.1335,...,2.652,2.863,2.667,2.492,2.270,1.669,2.288,2.825,2.376,2.610
9,9,1038650,30.4305,32.9715,0.0,82.6067,95.7199,91.7314,92.5541,86.4249,...,2.553,1.968,2.822,2.460,2.325,2.309,2.210,2.721,2.543,2.484
10,10,1070217,30.3028,26.1140,0.0,83.1401,101.9138,93.4935,95.9066,88.3465,...,2.872,2.789,3.281,2.442,2.776,2.210,2.408,2.919,2.579,2.915


In [32]:
#for df4
df_4["SubjectID"] = df_4["SubjectID"].astype(int)
combined_df["SubjectID"] = combined_df["SubjectID"].astype(int)

combined_df = combined_df.loc[:, ~combined_df.columns.str.contains("^index", case=False)]
df_4 = df_4.loc[:, ~df_4.columns.str.contains("^index", case=False)]

combined_df = combined_df.merge(df_4, on="SubjectID", how="left")



#for df5
df_5["SubjectID"] = df_5["SubjectID"].astype(int)
combined_df["SubjectID"] = combined_df["SubjectID"].astype(int)

combined_df = combined_df.loc[:, ~combined_df.columns.str.contains("^index", case=False)]
df_5 = df_5.loc[:, ~df_5.columns.str.contains("^index", case=False)]

combined_df_5 = combined_df.merge(df_5, on="SubjectID", how="left")
#combined_df_5.head(50)


pd.set_option('display.max_columns', None)
combined_df_5


,SubjectID,3rd-Ventricle,4th-Ventricle,5th-Ventricle,Brain-Stem,CC_Anterior,CC_Central,CC_Mid_Anterior,CC_Mid_Posterior,CC_Posterior,CSF,Left-Accumbens-area,Left-Amygdala,Left-Caudate,Left-Cerebellum-Cortex,Left-Cerebellum-White-Matter,Left-Hippocampus,Left-Inf-Lat-Vent,Left-Lateral-Ventricle,Left-Pallidum,Left-Putamen,Left-VentralDC,Left-WM-hypointensities,Left-choroid-plexus,Left-non-WM-hypointensities,Left-vessel,Optic-Chiasm,Right-Accumbens-area,Right-Amygdala,Right-Caudate,Right-Cerebellum-Cortex,Right-Cerebellum-White-Matter,Right-Hippocampus,Right-Inf-Lat-Vent,Right-Lateral-Ventricle,Right-Pallidum,Right-Putamen,Right-VentralDC,Right-WM-hypointensities,Right-choroid-plexus,Right-non-WM-hypointensities,Right-vessel,WM-hypointensities,non-WM-hypointensities,lh_G_front_inf-Opercular_thickness,lh_G_front_inf-Triangul_thickness,lh_G_front_middle_thickness,lh_G_front_sup_thickness,lh_G_oc-temp_lat-fusifor_thickness,lh_G_oc-temp_med-Lingual_thickness,lh_G_oc-temp_med-Parahip_thickness,lh_G_occipital_middle_thickness,lh_G_orbital_thickness,lh_G_pariet_inf-Angular_thickness,lh_G_pariet_inf-Supramar_thickness,lh_G_parietal_sup_thickness,lh_G_postcentral_thickness,lh_G_precentral_thickness,lh_G_rectus_thickness,lh_G_temp_sup-Lateral_thickness,lh_G_temp_sup-Plan_tempo_thickness,lh_G_temporal_inf_thickness,lh_G_temporal_middle_thickness,lh_Pole_occipital_thickness,lh_Pole_temporal_thickness,lh_S_calcarine_thickness,lh_S_central_thickness,lh_S_collat_transv_ant_thickness,lh_S_front_inf_thickness,lh_S_front_middle_thickness,lh_S_front_sup_thickness,lh_S_occipital_ant_thickness,lh_S_orbital-H_Shaped_thickness,lh_S_pericallosal_thickness,lh_S_postcentral_thickness,lh_S_precentral-inf-part_thickness,lh_S_precentral-sup-part_thickness,lh_S_temporal_inf_thickness,lh_S_temporal_sup_thickness,rh_G_cuneus_thickness,rh_G_front_middle_thickness,rh_G_front_sup_thickness,rh_G_oc-temp_lat-fusifor_thickness,rh_G_occipital_middle_thickness,rh_G_occipital_sup_thickness,rh_G_orbital_thickness,rh_G_pariet_inf-Angular_thickness,rh_G_pariet_inf-Supramar_thickness,rh_G_parietal_sup_thickness,rh_G_postcentral_thickness,rh_G_precentral_thickness,rh_G_precuneus_thickness,rh_G_temp_sup-Lateral_thickness,rh_G_temp_sup-Plan_tempo_thickness,rh_G_temporal_inf_thickness,rh_G_temporal_middle_thickness,rh_Pole_occipital_thickness,rh_S_calcarine_thickness,rh_S_central_thickness,rh_S_cingul-Marginalis_thickness,rh_S_front_inf_thickness,rh_S_front_middle_thickness,rh_S_front_sup_thickness,rh_S_interm_prim-Jensen_thickness,rh_S_oc-temp_lat_thickness,rh_S_occipital_ant_thickness,rh_S_parieto_occipital_thickness,rh_S_pericallosal_thickness,rh_S_postcentral_thickness,rh_S_precentral-inf-part_thickness,rh_S_precentral-sup-part_thickness,rh_S_temporal_sup_thickness,Year_scan_initial,Assessment_centre_initial,Month_scan_initial,Sex,Birthyear
0,1533138,36.8674,31.3818,0.0,81.5837,100.2311,94.2112,93.9213,89.7624,101.8197,50.2793,78.4219,71.2338,80.8676,59.8507,84.6188,68.5732,46.7229,33.9679,101.8322,86.8691,91.6590,0.0,51.1266,0.0,57.8200,73.7333,78.6486,67.9744,81.4864,60.6036,85.0926,68.6901,47.5164,34.8871,98.3383,83.9349,88.3665,0.0,52.4355,0.0,60.0000,72.3171,0.0,2.955,2.859,2.994,3.053,3.310,2.248,2.847,2.846,3.016,2.873,3.261,2.815,2.368,2.955,3.441,3.265,2.718,3.326,3.215,2.260,3.297,2.075,2.159,3.082,2.661,2.565,2.774,2.719,3.068,2.288,2.521,2.617,2.518,2.518,2.803,2.051,2.741,2.932,3.033,2.811,2.532,3.070,2.830,2.593,2.522,2.183,2.853,2.670,3.282,2.650,3.318,3.051,2.059,2.016,1.869,2.466,2.446,2.393,2.753,2.170,2.914,2.665,2.533,1.965,2.227,2.672,2.561,2.735,2010.180822,11016.0,3.0,0.0,1950.0
1,1875529,31.3638,30.1527,0.0,83.1511,95.3329,89.9717,90.3416,85.7553,100.0998,47.6822,75.2107,67.8566,79.1048,61.6535,86.0415,70.1340,41.7886,28.0750,101.7199,84.5846,92.3921,0.0,50.0821,0.0,56.6591,76.4964,78.8506,69.8590,81.3028,61.3760,85.2568,70.4684,42.9726,31.5494,101.0202,83.6074,90.9502,0.0,52.0951,0.0,60.4118,70.6537,0.0,3.153,3.050,2.993,3.167,3.214,2.035,3.097,2.610

In [34]:
combined_df_5["Year_scan_initial"] = combined_df_5["Year_scan_initial"].fillna(-1)
combined_df_5["Year_scan_initial"] = combined_df_5["Year_scan_initial"].astype(int)

combined_df_5["Birthyear"] = combined_df_5["Birthyear"].fillna(-1)
combined_df_5["Birthyear"] = combined_df_5["Birthyear"].astype(int)


combined_df_5["Age"]  = combined_df_5["Year_scan_initial"] - combined_df_5["Birthyear"]
combined_df_5

,SubjectID,3rd-Ventricle,4th-Ventricle,5th-Ventricle,Brain-Stem,CC_Anterior,CC_Central,CC_Mid_Anterior,CC_Mid_Posterior,CC_Posterior,CSF,Left-Accumbens-area,Left-Amygdala,Left-Caudate,Left-Cerebellum-Cortex,Left-Cerebellum-White-Matter,Left-Hippocampus,Left-Inf-Lat-Vent,Left-Lateral-Ventricle,Left-Pallidum,Left-Putamen,Left-VentralDC,Left-WM-hypointensities,Left-choroid-plexus,Left-non-WM-hypointensities,Left-vessel,Optic-Chiasm,Right-Accumbens-area,Right-Amygdala,Right-Caudate,Right-Cerebellum-Cortex,Right-Cerebellum-White-Matter,Right-Hippocampus,Right-Inf-Lat-Vent,Right-Lateral-Ventricle,Right-Pallidum,Right-Putamen,Right-VentralDC,Right-WM-hypointensities,Right-choroid-plexus,Right-non-WM-hypointensities,Right-vessel,WM-hypointensities,non-WM-hypointensities,lh_G_front_inf-Opercular_thickness,lh_G_front_inf-Triangul_thickness,lh_G_front_middle_thickness,lh_G_front_sup_thickness,lh_G_oc-temp_lat-fusifor_thickness,lh_G_oc-temp_med-Lingual_thickness,lh_G_oc-temp_med-Parahip_thickness,lh_G_occipital_middle_thickness,lh_G_orbital_thickness,lh_G_pariet_inf-Angular_thickness,lh_G_pariet_inf-Supramar_thickness,lh_G_parietal_sup_thickness,lh_G_postcentral_thickness,lh_G_precentral_thickness,lh_G_rectus_thickness,lh_G_temp_sup-Lateral_thickness,lh_G_temp_sup-Plan_tempo_thickness,lh_G_temporal_inf_thickness,lh_G_temporal_middle_thickness,lh_Pole_occipital_thickness,lh_Pole_temporal_thickness,lh_S_calcarine_thickness,lh_S_central_thickness,lh_S_collat_transv_ant_thickness,lh_S_front_inf_thickness,lh_S_front_middle_thickness,lh_S_front_sup_thickness,lh_S_occipital_ant_thickness,lh_S_orbital-H_Shaped_thickness,lh_S_pericallosal_thickness,lh_S_postcentral_thickness,lh_S_precentral-inf-part_thickness,lh_S_precentral-sup-part_thickness,lh_S_temporal_inf_thickness,lh_S_temporal_sup_thickness,rh_G_cuneus_thickness,rh_G_front_middle_thickness,rh_G_front_sup_thickness,rh_G_oc-temp_lat-fusifor_thickness,rh_G_occipital_middle_thickness,rh_G_occipital_sup_thickness,rh_G_orbital_thickness,rh_G_pariet_inf-Angular_thickness,rh_G_pariet_inf-Supramar_thickness,rh_G_parietal_sup_thickness,rh_G_postcentral_thickness,rh_G_precentral_thickness,rh_G_precuneus_thickness,rh_G_temp_sup-Lateral_thickness,rh_G_temp_sup-Plan_tempo_thickness,rh_G_temporal_inf_thickness,rh_G_temporal_middle_thickness,rh_Pole_occipital_thickness,rh_S_calcarine_thickness,rh_S_central_thickness,rh_S_cingul-Marginalis_thickness,rh_S_front_inf_thickness,rh_S_front_middle_thickness,rh_S_front_sup_thickness,rh_S_interm_prim-Jensen_thickness,rh_S_oc-temp_lat_thickness,rh_S_occipital_ant_thickness,rh_S_parieto_occipital_thickness,rh_S_pericallosal_thickness,rh_S_postcentral_thickness,rh_S_precentral-inf-part_thickness,rh_S_precentral-sup-part_thickness,rh_S_temporal_sup_thickness,Year_scan_initial,Assessment_centre_initial,Month_scan_initial,Sex,Birthyear,Age
0,1533138,36.8674,31.3818,0.0,81.5837,100.2311,94.2112,93.9213,89.7624,101.8197,50.2793,78.4219,71.2338,80.8676,59.8507,84.6188,68.5732,46.7229,33.9679,101.8322,86.8691,91.6590,0.0,51.1266,0.0,57.8200,73.7333,78.6486,67.9744,81.4864,60.6036,85.0926,68.6901,47.5164,34.8871,98.3383,83.9349,88.3665,0.0,52.4355,0.0,60.0000,72.3171,0.0,2.955,2.859,2.994,3.053,3.310,2.248,2.847,2.846,3.016,2.873,3.261,2.815,2.368,2.955,3.441,3.265,2.718,3.326,3.215,2.260,3.297,2.075,2.159,3.082,2.661,2.565,2.774,2.719,3.068,2.288,2.521,2.617,2.518,2.518,2.803,2.051,2.741,2.932,3.033,2.811,2.532,3.070,2.830,2.593,2.522,2.183,2.853,2.670,3.282,2.650,3.318,3.051,2.059,2.016,1.869,2.466,2.446,2.393,2.753,2.170,2.914,2.665,2.533,1.965,2.227,2.672,2.561,2.735,2010,11016.0,3.0,0.0,1950,60
1,1875529,31.3638,30.1527,0.0,83.1511,95.3329,89.9717,90.3416,85.7553,100.0998,47.6822,75.2107,67.8566,79.1048,61.6535,86.0415,70.1340,41.7886,28.0750,101.7199,84.5846,92.3921,0.0,50.0821,0.0,56.6591,76.4964,78.8506,69.8590,81.3028,61.3760,85.2568,70.4684,42.9726,31.5494,101.0202,83.6074,90.9502,0.0,52.0951,0.0,60.4118,70.6537,0.0,3.153,3.050,2.993,3.167,3.214,2.035,3.097,2.610,3

In [35]:
combined_df_5.to_csv("final_datatabel.csv")

In [41]:
#final clean up of the resulting dataset
df = pd.read_csv("final_datatabel.csv")
df.drop(columns=["Month_scan_initial","Unnamed: 0"],inplace=True)

df.rename(columns={'Year_scan_initial': 'Year_initial_scan', 'Assessment_centre_initial': 'Site'}, inplace=True)
df.set_index("SubjectID")

,3rd-Ventricle,4th-Ventricle,5th-Ventricle,Brain-Stem,CC_Anterior,CC_Central,CC_Mid_Anterior,CC_Mid_Posterior,CC_Posterior,CSF,Left-Accumbens-area,Left-Amygdala,Left-Caudate,Left-Cerebellum-Cortex,Left-Cerebellum-White-Matter,Left-Hippocampus,Left-Inf-Lat-Vent,Left-Lateral-Ventricle,Left-Pallidum,Left-Putamen,Left-VentralDC,Left-WM-hypointensities,Left-choroid-plexus,Left-non-WM-hypointensities,Left-vessel,Optic-Chiasm,Right-Accumbens-area,Right-Amygdala,Right-Caudate,Right-Cerebellum-Cortex,Right-Cerebellum-White-Matter,Right-Hippocampus,Right-Inf-Lat-Vent,Right-Lateral-Ventricle,Right-Pallidum,Right-Putamen,Right-VentralDC,Right-WM-hypointensities,Right-choroid-plexus,Right-non-WM-hypointensities,Right-vessel,WM-hypointensities,non-WM-hypointensities,lh_G_front_inf-Opercular_thickness,lh_G_front_inf-Triangul_thickness,lh_G_front_middle_thickness,lh_G_front_sup_thickness,lh_G_oc-temp_lat-fusifor_thickness,lh_G_oc-temp_med-Lingual_thickness,lh_G_oc-temp_med-Parahip_thickness,lh_G_occipital_middle_thickness,lh_G_orbital_thickness,lh_G_pariet_inf-Angular_thickness,lh_G_pariet_inf-Supramar_thickness,lh_G_parietal_sup_thickness,lh_G_postcentral_thickness,lh_G_precentral_thickness,lh_G_rectus_thickness,lh_G_temp_sup-Lateral_thickness,lh_G_temp_sup-Plan_tempo_thickness,lh_G_temporal_inf_thickness,lh_G_temporal_middle_thickness,lh_Pole_occipital_thickness,lh_Pole_temporal_thickness,lh_S_calcarine_thickness,lh_S_central_thickness,lh_S_collat_transv_ant_thickness,lh_S_front_inf_thickness,lh_S_front_middle_thickness,lh_S_front_sup_thickness,lh_S_occipital_ant_thickness,lh_S_orbital-H_Shaped_thickness,lh_S_pericallosal_thickness,lh_S_postcentral_thickness,lh_S_precentral-inf-part_thickness,lh_S_precentral-sup-part_thickness,lh_S_temporal_inf_thickness,lh_S_temporal_sup_thickness,rh_G_cuneus_thickness,rh_G_front_middle_thickness,rh_G_front_sup_thickness,rh_G_oc-temp_lat-fusifor_thickness,rh_G_occipital_middle_thickness,rh_G_occipital_sup_thickness,rh_G_orbital_thickness,rh_G_pariet_inf-Angular_thickness,rh_G_pariet_inf-Supramar_thickness,rh_G_parietal_sup_thickness,rh_G_postcentral_thickness,rh_G_precentral_thickness,rh_G_precuneus_thickness,rh_G_temp_sup-Lateral_thickness,rh_G_temp_sup-Plan_tempo_thickness,rh_G_temporal_inf_thickness,rh_G_temporal_middle_thickness,rh_Pole_occipital_thickness,rh_S_calcarine_thickness,rh_S_central_thickness,rh_S_cingul-Marginalis_thickness,rh_S_front_inf_thickness,rh_S_front_middle_thickness,rh_S_front_sup_thickness,rh_S_interm_prim-Jensen_thickness,rh_S_oc-temp_lat_thickness,rh_S_occipital_ant_thickness,rh_S_parieto_occipital_thickness,rh_S_pericallosal_thickness,rh_S_postcentral_thickness,rh_S_precentral-inf-part_thickness,rh_S_precentral-sup-part_thickness,rh_S_temporal_sup_thickness,Year_initial_scan,Site,Sex,Birthyear,Age
SubjectID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1533138,36.8674,31.3818,0.0,81.5837,100.2311,94.2112,93.9213,89.7624,101.8197,50.2793,78.4219,71.2338,80.8676,59.8507,84.6188,68.5732,46.7229,33.9679,101.8322,86.8691,91.6590,0.0,51.1266,0.0,57.8200,73.7333,78.6486,67.9744,81.4864,60.6036,85.0926,68.6901,47.5164,34.8871,98.3383,83.9349,88.3665,0.0,52.4355,0.0,60.0000,72.3171,0.0,2.955,2.859,2.994,3.053,3.310,2.248,2.847,2.846,3.016,2.873,3.261,2.815,2.368,2.955,3.441,3.265,2.718,3.326,3.215,2.260,3.297,2.075,2.159,3.082,2.661,2.565,2.774,2.719,3.068,2.288,2.521,2.617,2.518,2.518,2.803,2.051,2.741,2.932,3.033,2.811,2.532,3.070,2.830,2.593,2.522,2.183,2.853,2.670,3.282,2.650,3.318,3.051,2.059,2.016,1.869,2.466,2.446,2.393,2.753,2.170,2.914,2.665,2.533,1.965,2.227,2.672,2.561,2.735,2010,11016.0,0.0,1950,60
1875529,31.3638,30.1527,0.0,83.1511,95.3329,89.9717,90.3416,85.7553,100.0998,47.6822,75.2107,67.8566,79.1048,61.6535,86.0415,70.1340,41.7886,28.0750,101.7199,84.5846,92.3921,0.0,50.0821,0.0,56.6591,76.4964,78.8506,69.8590,81.3028,61.3760,85.2568,70.4684,42.9726,31.5494,101.0202,83.6074,90.9502,0.0,52.0951,0.0,6

In [42]:
df.to_csv("Final_Data.csv")

In [16]:
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"

data_4 = os.path.join(csv_path, "Yr_mnth_site_scan.csv")
df_4 = pd.read_csv(data_4)
df_4 = df_4.rename(columns={"Assessment_centre_1_repeat" : "Site_imaging"})
df_5 = df_4[["SubjectID", "Site_imaging"]].copy()
df_5


#let's see if this merges with the other data

,SubjectID,Site_imaging
0,1000012,NaN
1,1000029,NaN
2,1000031,NaN
3,1000047,NaN
4,1000050,11025.0
...,...,...
502361,6025263,NaN
502362,6025278,NaN
502363,6025280,NaN
502364,6025291,NaN


In [22]:
df = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Final_Data.csv")
df_check = df.merge(df_5, how="left")
df_check = df_check.drop(columns=["Site"])
df_check = df_check.rename(columns={"Site_imaging":"Site"})
#df_check.to_csv("Final_data_corrected_site.csv")
df_check

,Unnamed: 0,SubjectID,3rd-Ventricle,4th-Ventricle,5th-Ventricle,Brain-Stem,CC_Anterior,CC_Central,CC_Mid_Anterior,CC_Mid_Posterior,...,rh_S_pericallosal_thickness,rh_S_postcentral_thickness,rh_S_precentral-inf-part_thickness,rh_S_precentral-sup-part_thickness,rh_S_temporal_sup_thickness,Year_initial_scan,Sex,Birthyear,Age,Site
0,0,1533138,36.8674,31.3818,0.0,81.5837,100.2311,94.2112,93.9213,89.7624,...,1.965,2.227,2.672,2.561,2.735,2010,0.0,1950,60,11025.0
1,1,1875529,31.3638,30.1527,0.0,83.1511,95.3329,89.9717,90.3416,85.7553,...,2.407,2.016,2.794,2.706,2.676,2008,0.0,1948,60,11025.0
2,2,4758824,35.0732,36.2819,0.0,84.1654,99.6201,95.3489,95.4416,93.4878,...,1.793,2.397,2.537,2.547,2.684,2008,0.0,1946,62,11025.0
3,3,5304207,41.5897,33.3825,0.0,84.4920,104.1299,93.2735,95.2359,95.4301,...,1.782,2.377,2.501,2.696,2.746,2010,0.0,1963,47,11025.0
4,4,2747983,42.9063,33.2600,0.0,83.8396,103.1391,92.3403,90.9298,94.4794,...,1.927,2.579,2.863,2.843,2.829,2008,0.0,1963,45,11025.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33937,33937,3463731,35.1942,33.5758,0.0,84.6992,102.1988,93.7001,92.8186,93.6035,...,1.648,2.351,2.870,2.696,2.851,2008,0.0,1957,51,11027.0
33938,33938,3814047,36.9796,33.8569,0.0,82.1534,100.7485,96.7414,94.8558,92.6987,...,1.963,2.245,2.408,2.523,2.536,2007,1.0,1960,47,11025.0
33939,33939,4851516,31.9402,32.7079,0.0,84.1758,98.3044,92.3463,91.8287,87.2853,...,2.461,2.173,2.552,2.408,2.845,2009,1.0,1940,69,11027.0
33940,33940,3626816,42.8421,34.9272,0.0,84.0678,105.3313,94.4410,91.4660,91.4544,...,2.071,2.475,2.698,2.665,3.171,2008,1.0,1961,47,11025.0


In [ ]:
health_outcomes = pd.read_csv("/project_cephfs/3022017.06/UKB/phenotypes/current/50_health_outcomes.csv")
health_outcomes.head()